# ISOT Drone Dataset Preprocessing
**Source:** University of Victoria ISOT Lab  
**Attack Types:** DoS, Injection, Ip Spoofing, Manipulation, MITM, Password Cracking, Replay, Unauth, Video, Benign  
**Output:** X_train.npy, X_test.npy, y_train.csv, y_test.csv, feature_names.csv, label_classes.csv

## Dataset Domain Knowledge — ISOT Drone Network Traffic

### How Drone Network Communication Works
Drones communicate with Ground Control Stations (GCS) over WiFi/radio.
This external network traffic can be captured and analyzed for attacks.
ISOT captures this network traffic and labels it by attack type.
Features are extracted from network packets — similar to CICIDS2017.

### What Each Column Means

| Column | Meaning | Attack Signal |
|--------|---------|---------------|
| `Payload_Length` | Size of network packet | Very large/small = abnormal |
| `Rate` | Packets per second | Very high = DoS or Flooding |
| `Srate, Drate` | Send/receive rate | Imbalanced = one-sided attack |
| `fin_flag, syn_flag` | TCP connection flags | Unusual counts = scanning |
| `Entropy` | Randomness of packet content | High = encrypted attack traffic |
| `Duration` | How long the flow lasts | Very short = attack probing |
| `Protocol Type` | Network protocol used | Unusual protocol = injection |

### The 9 Attack Types

**DoS** — overwhelm attack
- Floods drone network with traffic until it stops responding

**Injection** — command attack
- Injects fake control commands into the communication stream

**IP Spoofing** — identity attack
- Fakes the source IP address to impersonate trusted devices

**Manipulation** — data tampering
- Modifies legitimate packets in transit

**MITM** — interception attack
- Attacker sits between drone and GCS, reads and modifies all traffic

**Password Cracking** — credential attack
- Tries many passwords to gain access to drone control system

**Replay** — copy-paste attack
- Records legitimate commands and replays them later

**Unauth** — unauthorized access
- Connects to drone systems without proper credentials

**Video** — surveillance attack
- Intercepts drone camera video feed

### Expected XAI Findings
- **SHAP on Rate** → key for DoS detection (very high packet rate)
- **SHAP on Entropy** → key for encrypted attack traffic
- **LIME** → local explanation per packet, may focus on different features for minority attacks
- **Permutation Importance** → confirms which features cause biggest F1 drop when shuffled
- **Class imbalance effect** → all 3 XAI methods biased toward Benign and DoS majority classes
- **Minority attacks** (Injection, Replay, Unauth) may have unreliable XAI explanations
- **Key question** → do SHAP, LIME, and PI agree across all 10 attack types?

In [22]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [23]:
# what cell 3 for ? it is visit nultiple file holder to read those file then create a big table
# why: when model trail it, only need table instead of big holders. 
# How:
# Step 1: Go to DoS folder
# Step 2: Find all CSV files inside
# Step 3: Read each file → add label "DoS"
# Step 4: Go to next folder (Injection)
# Step 5: Repeat until all 10 folders done
# Step 6: Stack everything into one big table

# base path to extracted ISOT Drone Dataset CSV files
# each subfolder represents one attack type — folder name is used as the ground truth label
base_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/ISOT/ISOT Drone Dataset/Dataset/new_feature_csv/"

# map folder names to standardized label names
# ISOT does not include a label column in the CSV files —
# the attack type is encoded in the folder structure instead.
# This is a common pattern in network traffic datasets captured per-scenario.
label_map = {
    'Regular':          'Benign',
    'DoS':              'DoS',
    'Injection':        'Injection',
    'Ip Spoofing':      'Ip_Spoofing',
    'Manipulation':     'Manipulation',
    'MITM':             'MITM',
    'Password Cracking':'Password_Cracking',
    'Replay':           'Replay',
    'Unauth':           'Unauth',
    'Video':            'Video'
}

# collect all individual file dataframes before concatenating
# loading separately first allows us to assign labels per folder before merging
all_dfs = [] # create a empty table 

# .tiem() -> make folder name and label at same time
for folder, label in label_map.items():
    folder_path = os.path.join(base_path, folder) # Builds the full folder address by joining two pieces: i.g.: base_path  = ".../new_feature_csv/"
    for filename in os.listdir(folder_path): # skip any file is NOT a csv
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename) # build a full path for specific files: i.g.: result = ".../new_feature_csv/DoS/Dos_1_4_30_20mins.pcap.csv"
            df_temp = pd.read_csv(file_path) # Opens that one CSV file and loads it into a temporary table called df_temp. Like opening one Excel sheet.
            
            # assign ground truth label from folder name —
            # this is the labeling methodology for ISOT since no label column exists in raw files
            df_temp['label'] = label # add new cumlun called: label
            all_dfs.append(df_temp) # add the lable culumn on table

# concatenate all files into one unified dataframe
# ignore_index=True resets row numbers to a continuous sequence across all files
df = pd.concat(all_dfs, ignore_index=True) # combine all together to be on edocument

print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"\nLabel counts:\n{df['label'].value_counts()}")

# ============================================================
# CLASS IMBALANCE OBSERVATION — IMPORTANT FINDING
# ============================================================
# Results show severe class imbalance across 10 attack types:
#
#   Benign + DoS + Password_Cracking + MITM = ~96.4% of all traffic
#   Remaining 6 attack types (Ip_Spoofing, Manipulation, Replay,
#   Unauth, Injection, Video) = less than 4% combined
#
# This means the model will spend ~96% of its learning on 4 classes
# and less than 4% on the remaining 6 attack types.
#
# RISK FOR DRONE SECURITY:
# The 6 minority attack types (e.g. Injection, Manipulation, Replay)
# are dangerous real-world threats. If the model ignores them due to
# low representation, an attacker exploiting these vectors could
# cause serious drone failure — navigation errors, hijacking,
# or loss of control — without being detected.
#
# DECISION: We do NOT fix the imbalance (no oversampling/undersampling)
# because:
# 1. Consistency — CICIDS2017 baseline was not rebalanced
# 2. Realism — this distribution reflects real drone network traffic
# 3. Research contribution — imbalance effect on XAI explanations
#    is itself a key finding of this benchmark
#
# Near-zero F1 scores for minority classes in results are EXPECTED
# and will be documented and discussed in the paper.
# ============================================================

Total rows: 2945986
Total columns: 63

Label counts:
label
Benign               1280814
DoS                  1220131
Password_Cracking     231218
MITM                  105141
Video                  77235
Ip_Spoofing             9733
Manipulation            6099
Replay                  5860
Unauth                  5012
Injection               4743
Name: count, dtype: int64


In [8]:
# Cell 4 (find invalid information) removes anything that is not a clean, real, useful number for the model to learn from.
# drop 'ts' (timestamp) column — timestamps are not behavioral features
# and would introduce data leakage if used for detection.
# This follows standard IDS preprocessing practice.
df = df.drop(columns=['ts'], errors='ignore')

# check for missing values (NaN) — empty cells caused by incomplete packet captures
# or merging files with different column counts across attack scenarios
nan_count = df.isnull().sum().sum()
print(f"Total NaN values: {nan_count}") # NaN = Not a Number = empty cell.

# check for infinite values (Inf) — can occur during feature calculation
# e.g. division by zero in rate or ratio features
inf_count = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
print(f"Total Inf values: {inf_count}")

print(f"\nShape after dropping ts: {df.shape}")

Total NaN values: 7893
Total Inf values: 0

Shape after dropping ts: (2945986, 62)


In [9]:
# Cell 5: delete the invalue information from cell 4
# drop all rows that contain any NaN value
# 7,893 NaN rows = only 0.27% of total data — safe to remove without losing meaningful information
# keeping NaN rows would cause errors during model training
df = df.dropna()

# verify NaN is fully removed
nan_check = df.isnull().sum().sum()
print(f"NaN values remaining: {nan_check}")

# confirm how many rows were removed
print(f"Rows removed: {2945986 - df.shape[0]}")
print(f"Rows remaining: {df.shape[0]}")
print(f"Final shape: {df.shape}")


NaN values remaining: 0
Rows removed: 7839
Rows remaining: 2938147
Final shape: (2938147, 62)


In [27]:
# Cell 5: delete the invalue information from cell 4
# drop all rows that contain any NaN value
# 7,893 NaN rows = only 0.27% of total data — safe to remove without losing meaningful information
# keeping NaN rows would cause errors during model training
df = df.dropna()

# verify NaN is fully removed
nan_check = df.isnull().sum().sum()
print(f"NaN values remaining: {nan_check}")

# confirm how many rows were removed
print(f"Rows removed: {2945986 - df.shape[0]}")
print(f"Rows remaining: {df.shape[0]}")
print(f"Final shape: {df.shape}")

NaN values remaining: 0
Rows removed: 7839
Rows remaining: 2938147
Final shape: (2938147, 63)


In [11]:
# Cell 6: convert the text to number to make the scalling are equally to training.
# Benign           → 0
# DoS              → 1
# Injection        → 2
# Ip_Spoofing      → 3
# Manipulation     → 4
# MITM             → 5
# Password_Cracking→ 6
# Replay           → 7
# Unauth           → 8
# Video            → 9

# separate features (X) and label (y)
# X = all columns except label — these are the input to the model
# y = label column only — this is what the model learns to predict
X = df.drop(columns=['label'])
y = df['label']

# encode text labels into numbers — models only understand numbers not text
# LabelEncoder assigns numbers alphabetically
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# print mapping so we know which number = which attack type
print("Label encoding:")
for i, label in enumerate(le.classes_):
    print(f"  {label} → {i}")

# scale all features to same range using StandardScaler
# prevents large-value features from dominating small-value features during training
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nFeatures shape: {X_scaled.shape}")
print(f"Labels shape: {y_encoded.shape}")

Label encoding:
  Benign → 0
  DoS → 1
  Injection → 2
  Ip_Spoofing → 3
  MITM → 4
  Manipulation → 5
  Password_Cracking → 6
  Replay → 7
  Unauth → 8
  Video → 9

Features shape: (2938147, 61)
Labels shape: (2938147,)


In [ ]:
# Cell 7: what: combine everthing back together and svaes it as one clean CSV file on the disk
# why: 
# X_scaled  = 61 feature columns (numpy array)
# y_encoded = 1 label column (numpy array)
# the model notebook need one single file with features and labels together for save in disk. preprocess once once. CNN,RF, AE will use those file
# How: 
# Step 1: Convert X_scaled (numpy array) back to 
#        a proper dataframe with column names
# Step 2: Add y_encoded as the 'label' column
# Step 3: Save as processed_ISOT.csv to disk
# Step 4: Print confirmation — shape and label counts

# combine scaled features and encoded labels back into one dataframe
# X_scaled is a numpy array — convert back to dataframe with original column names
# separate features and labels from sampled dataframe
X_sample = df_sampled.drop(columns=['label'])
y_sample = df_sampled['label']

# encode labels and scale features on sampled data
le = LabelEncoder()
y_encoded_sample = le.fit_transform(y_sample)

scaler = StandardScaler()
X_scaled_sample = scaler.fit_transform(X_sample)

# combine back into one dataframe
df_processed = pd.DataFrame(X_scaled_sample, columns=X_sample.columns)
df_processed['label'] = y_encoded_sample

# save to disk
save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/ISOT/processed_ISOT.csv"
df_processed.to_csv(save_path, index=False)

print(f"Saved: {save_path}")
print(f"Final shape: {df_processed.shape}")
print(f"\nLabel distribution:")
print(df_processed['label'].value_counts())

In [13]:
# reduce the size for save room in disk for cell 7 process
# sample 10% of each label to reduce file size
# stratify by label to keep same proportion of each attack type
df_sampled = df.groupby('label').apply(
    lambda x: x.sample(frac=0.1, random_state=42)
).reset_index(drop=True)

print(f"Sampled rows: {df_sampled.shape[0]}")
print(f"\nLabel counts after sampling:\n{df_sampled['label'].value_counts()}")

/var/folders/28/cggd8l710jz37nj7nfdl032r0000gn/T/ipykernel_26256/2413266986.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sampled = df.groupby('label').apply(


Sampled rows: 293816

Label counts after sampling:
label
Benign               128069
DoS                  121244
Password_Cracking     23122
MITM                  10514
Video                  7723
Ip_Spoofing             973
Manipulation            610
Replay                  586
Unauth                  501
Injection               474
Name: count, dtype: int64


In [26]:
# Split train/test 70/30
# stratify = keeps same class ratio in both train and test sets
# random_state=42 = reproducible split every run
from sklearn.model_selection import train_test_split

# use X_scaled_sample — the 10% sampled version
# using full 2.9M rows would exceed disk space
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_sample, y_encoded_sample,
    test_size=0.3,
    random_state=42,
    stratify=y_encoded_sample
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (205671, 61)
X_test shape: (88145, 61)
y_train shape: (205671,)
y_test shape: (88145,)


In [21]:
import os
save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/ISOT/processed/"
os.makedirs(save_path, exist_ok=True)

np.save(save_path + "X_train.npy", X_train)
np.save(save_path + "X_test.npy", X_test)
pd.DataFrame(y_train).to_csv(save_path + "y_train.csv", index=False)
pd.DataFrame(y_test).to_csv(save_path + "y_test.csv", index=False)
pd.DataFrame(X_sample.columns).to_csv(save_path + "feature_names.csv", index=False)
pd.DataFrame(le.classes_).to_csv(save_path + "label_classes.csv", index=False)

print("Saved!")
print(f"X_train.npy: {X_train.shape}")
print(f"X_test.npy: {X_test.shape}")
print(f"y_train.csv: {y_train.shape}")
print(f"y_test.csv: {y_test.shape}")

Saved!
X_train.npy: (205671, 61)
X_test.npy: (88145, 61)
y_train.csv: (205671,)
y_test.csv: (88145,)


## Preprocessing Complete

| Item | Value |
|------|-------|
| Source | ISOT Drone Dataset — University of Victoria |
| Original rows | 2,938,147 |
| Sampled rows | 293,816 (10% stratified) |
| Total features | 61 |
| Labels | 10 attack types |
| Class imbalance | Yes — Benign+DoS = 84.9% |
| Train set | 205,671 rows |
| Test set | 88,145 rows |
| Split ratio | 70% train / 30% test |
| Output files | X_train.npy, X_test.npy, y_train.csv, y_test.csv |